In [1]:
import os
import sys
import requests
import pandas as pd
from dotenv import load_dotenv

sys.path.append("../src")
import sec_extract

load_dotenv()
HEADERS = {"User-Agent": os.getenv("SEC_USER_AGENT")}

url = "https://data.sec.gov/api/xbrl/companyfacts/CIK0001436126.json"
data = requests.get(url, headers=HEADERS, timeout=30).json()

mistras = sec_extract.extract_company(data, "1231")
print("Rows extracted:", len(mistras))
mistras[mistras["fiscal_year"] == 2025]

Rows extracted: 148


,fiscal_year,end,value_000s,tag,accn,variable
9,2025,2025-12-31,724024.0,RevenueFromContractWithCustomerExcludingAssess...,0001628280-26-016765,revenue
19,2025,2025-12-31,40572.0,OperatingIncomeLoss,0001628280-26-016765,operating_income
28,2025,2025-12-31,14643.0,InterestExpenseNonoperating,0001628280-26-016765,interest_expense
38,2025,2025-12-31,32058.0,WeightedAverageNumberOfDilutedSharesOutstanding,0001628280-26-016765,diluted_shares
48,2025,2025-12-31,31008.0,DepreciationDepletionAndAmortization,0001628280-26-016765,d_and_a
57,2025,2025-12-31,32981.0,NetCashProvidedByUsedInOperatingActivities,0001628280-26-016765,cfo
66,2025,2025-12-31,24674.0,PaymentsToAcquirePropertyPlantAndEquipment,0001628280-26-016765,capex
83,2025,2025-12-31,28008.0,CashAndCashEquivalentsAtCarryingValue,0001628280-26-016765,cash
93,2025,2025-12-31,12849.0,LongTermDebtCurrent,0001628280-26-016765,short_term_debt
103,2025,2025-12-31,165143.0,LongTermDebtNoncurrent,0001628280-26-016765,long_term_debt


In [2]:
answer_key_2025 = {
    "revenue": 724024, "operating_income": 40572, "interest_expense": 14643,
    "diluted_shares": 32058, "d_and_a": 31008, "cfo": 32981, "capex": 24674,
    "acquisitions_paid": 0, "cash": 28008, "short_term_debt": 12849,
    "long_term_debt": 165143, "finance_lease_current": 7025,
    "finance_lease_noncurrent": 17340, "stockholders_equity": 235104,
    "accounts_receivable": 154673, "goodwill": 184829,
}

fy2025 = mistras[mistras["fiscal_year"] == 2025].set_index("variable")["value_000s"]

for variable, expected in answer_key_2025.items():
    actual = fy2025.get(variable)
    if actual is None:
        status = "MISSING"
    elif abs(actual - expected) < 1:
        status = "MATCH"
    else:
        status = "MISMATCH"
    print(f"{variable:26} key: {expected:>8}   api: {actual}   {status}")

revenue                    key:   724024   api: 724024.0   MATCH
operating_income           key:    40572   api: 40572.0   MATCH
interest_expense           key:    14643   api: 14643.0   MATCH
diluted_shares             key:    32058   api: 32058.0   MATCH
d_and_a                    key:    31008   api: 31008.0   MATCH
cfo                        key:    32981   api: 32981.0   MATCH
capex                      key:    24674   api: 24674.0   MATCH
acquisitions_paid          key:        0   api: None   MISSING
cash                       key:    28008   api: 28008.0   MATCH
short_term_debt            key:    12849   api: 12849.0   MATCH
long_term_debt             key:   165143   api: 165143.0   MATCH
finance_lease_current      key:     7025   api: 7025.0   MATCH
finance_lease_noncurrent   key:    17340   api: 17340.0   MATCH
stockholders_equity        key:   235104   api: 235104.0   MATCH
accounts_receivable        key:   154673   api: 154673.0   MATCH
goodwill                   key:   1848

In [3]:
import json
import time

CACHE_DIR = "../data/raw/companyfacts"
os.makedirs(CACHE_DIR, exist_ok=True)

universe = pd.read_csv("../data/processed/universe_sic_filtered.csv")
print("Companies in universe:", len(universe))


def load_company_facts(cik):
    """Load a company's facts from the local cache, downloading only if not saved yet."""
    path = f"{CACHE_DIR}/CIK{cik:010d}.json"
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            return json.load(f)

    url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik:010d}.json"
    response = requests.get(url, headers=HEADERS, timeout=60)
    time.sleep(0.12)   # stay under the SEC's rate limit
    if response.status_code != 200:
        return None    # e.g. a company with no XBRL financial data

    with open(path, "w", encoding="utf-8") as f:
        f.write(response.text)
    return response.json()

Companies in universe: 167


In [4]:
all_tables = []
log = []

for i, company in enumerate(universe.itertuples(), start=1):
    cik, ticker = company.cik, company.ticker
    try:
        if pd.isna(company.fiscal_year_end):
            log.append([cik, ticker, "NO FISCAL YEAR END"])
            continue
        fye = f"{int(company.fiscal_year_end):04d}"   # e.g. 630 becomes "0630"

        data = load_company_facts(cik)
        if data is None:
            log.append([cik, ticker, "NO COMPANY FACTS"])
            continue

        table = sec_extract.extract_company(data, fye)
        if table is None:
            log.append([cik, ticker, "NO US-GAAP DATA"])
            continue

        table.insert(0, "cik", cik)
        table.insert(1, "ticker", ticker)
        all_tables.append(table)
        log.append([cik, ticker, "OK"])

    except Exception as error:
        log.append([cik, ticker, f"ERROR: {error}"])

    if i % 25 == 0 or i == len(universe):
        print(f"Processed {i} of {len(universe)}")

Processed 25 of 167
Processed 50 of 167
Processed 75 of 167
Processed 100 of 167
Processed 150 of 167
Processed 167 of 167


In [5]:
financials = pd.concat(all_tables, ignore_index=True)
financials.to_csv("../data/processed/financials_long.csv", index=False)

status = pd.DataFrame(log, columns=["cik", "ticker", "status"])
status.to_csv("../data/processed/extraction_log.csv", index=False)

print("Rows saved:", len(financials))
print("Companies with data:", financials["cik"].nunique())
status["status"].value_counts()

Rows saved: 15112
Companies with data: 100


status
OK                    100
ERROR: 'filed'         47
NO FISCAL YEAR END     11
NO US-GAAP DATA         9
Name: count, dtype: int64

In [6]:
   financials[(financials["ticker"] == "MG") & (financials["fiscal_year"] == 2025) & (financials["variable"] == "revenue")]

,cik,ticker,fiscal_year,end,value_000s,tag,accn,variable
10606,1436126,MG,2025,2025-12-31,724024.0,RevenueFromContractWithCustomerExcludingAssess...,0001628280-26-016765,revenue
